# Zero-Shot Evaluation on Kaggle Dataset

Evaluates `typeform/distilbert-base-uncased-mnli` on the `you-are-bot-2` dataset.

**Expected files** (download from Kaggle and place in `../data/`):
- `../data/train.json`
- `../data/ytrain.csv`

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    roc_auc_score,
)
from transformers import pipeline

DATA_DIR = Path("../data")
MODEL_NAME = "typeform/distilbert-base-uncased-mnli"
CANDIDATE_LABELS = ["bot", "human"]
HYPOTHESIS_TEMPLATE = "This message was written by a {}."

## 1. Load data

In [ ]:
with open(DATA_DIR / "train.json") as f:
    dialogs = json.load(f)

labels_df = pd.read_csv(DATA_DIR / "ytrain.csv")

print(f"Dialogs: {len(dialogs)}")
print(f"Labels:  {len(labels_df)}")
print(labels_df.head())

## 2. Build per-participant table

In [ ]:
def make_participant_table(dialogs: list, labels_df: pd.DataFrame) -> pd.DataFrame:
    label_lookup = {
        (str(r["dialog_id"]), int(r["participant_index"])): int(r["is_bot"])
        for _, r in labels_df.iterrows()
    }
    rows = []
    for dialog in dialogs:
        did = str(dialog["id"])
        messages = dialog.get("messages", [])
        speakers: dict[int, list[str]] = {}
        for msg in messages:
            idx = int(msg["participant_index"])
            speakers.setdefault(idx, []).append(str(msg["text"]))
        for idx, texts in speakers.items():
            key = (did, idx)
            if key not in label_lookup:
                continue
            rows.append({
                "dialog_id": did,
                "participant_index": idx,
                "speaker_text": " ".join(texts),
                "is_bot": label_lookup[key],
            })
    return pd.DataFrame(rows)


df = make_participant_table(dialogs, labels_df)
print(df.shape)
print(df["is_bot"].value_counts())
df.head()

## 3. Load zero-shot pipeline

In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME,
    device=-1,  # CPU; set to 0 for GPU
)
print("Model loaded.")

## 4. Run classification

Each participant is represented by the concatenation of all their messages.

In [ ]:
def predict_bot_probability(text: str) -> float:
    result = classifier(
        text,
        candidate_labels=CANDIDATE_LABELS,
        hypothesis_template=HYPOTHESIS_TEMPLATE,
    )
    bot_idx = result["labels"].index(CANDIDATE_LABELS[0])
    return float(np.clip(result["scores"][bot_idx], 0.0, 1.0))


# Truncate long texts to avoid token limits
MAX_CHARS = 512
texts = df["speaker_text"].str[:MAX_CHARS].tolist()

probas = []
for i, text in enumerate(texts):
    p = predict_bot_probability(text)
    probas.append(p)
    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{len(texts)} done")

df["is_bot_probability"] = probas
print("Done.")

## 5. Metrics

In [ ]:
y_true = df["is_bot"].values
y_prob = df["is_bot_probability"].values
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "ROC-AUC":    round(roc_auc_score(y_true, y_prob), 4),
    "Accuracy":   round(accuracy_score(y_true, y_pred), 4),
    "F1":         round(f1_score(y_true, y_pred), 4),
    "Log-loss":   round(log_loss(y_true, y_prob), 4),
    "Brier":      round(brier_score_loss(y_true, y_prob), 4),
}

pd.DataFrame([metrics])

## 6. Distribution of predicted probabilities

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for label, group in df.groupby("is_bot"):
    ax.hist(
        group["is_bot_probability"],
        bins=30,
        alpha=0.6,
        label="bot" if label == 1 else "human",
    )
ax.set_xlabel("is_bot_probability")
ax.set_ylabel("Count")
ax.set_title(f"Zero-shot ({MODEL_NAME})")
ax.legend()
plt.tight_layout()
plt.show()